In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ouro")

spark.sql("SELECT * FROM prata.eventos").createOrReplaceTempView("prata_eventos_view")
spark.sql("SELECT * FROM prata.eventos_presenca").createOrReplaceTempView("prata_presenca_view")

print("Schema e views criados.")

# Entrega 1: Tabela Gold de Eventos
Eventos com dimensões: órgão, tipo de evento e data. Pronta para consumo analítico.


In [0]:
%sql
CREATE OR REPLACE TABLE ouro.eventos_gold
USING DELTA
COMMENT 'Camada Ouro - Tabela gold de eventos com dimensoes'
AS
SELECT 
    id_evento,
    data_hora_inicio,
    data_hora_fim,
    situacao,
    descricao_tipo,
    descricao,
    local_externo,
    url_registro,
    sigla_orgao,
    nome_orgao,
    cod_tipo_orgao,
    local_camara_nome,
    ano,
    mes,
    dia_semana,
    semana_ano,
    chave_semana,
    data_coleta,
    origem
FROM prata_eventos_view

In [0]:
%sql
SELECT 
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT descricao_tipo) AS tipos_evento,
    COUNT(DISTINCT sigla_orgao) AS orgaos,
    COUNT(DISTINCT chave_semana) AS semanas,
    MIN(data_hora_inicio) AS primeiro,
    MAX(data_hora_inicio) AS ultimo
FROM ouro.eventos_gold

# Entrega 2: Taxa de Presença por Deputado e Tipo de Evento ao Longo do Ano
Presença em 2024 agrupada por deputado, tipo de evento e mês.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.taxa_presenca
USING DELTA
COMMENT 'Camada Ouro - Taxa de presenca por deputado e tipo de evento ao longo do ano'
AS
WITH eventos_por_tipo_ano AS (
    SELECT 
        descricao_tipo,
        ano,
        COUNT(*) AS total_eventos_tipo
    FROM prata_eventos_view
    WHERE ano = 2024
    GROUP BY descricao_tipo, ano
),
presenca_por_deputado AS (
    SELECT 
        id_deputado,
        nome_deputado,
        sigla_partido,
        sigla_uf,
        descricao_tipo,
        ano,
        COUNT(DISTINCT id_evento) AS eventos_participados
    FROM prata_presenca_view
    WHERE ano = 2024
    GROUP BY id_deputado, nome_deputado, sigla_partido, sigla_uf, descricao_tipo, ano
)
SELECT 
    p.id_deputado,
    p.nome_deputado,
    p.sigla_partido,
    p.sigla_uf,
    p.descricao_tipo,
    p.eventos_participados,
    e.total_eventos_tipo,
    ROUND(p.eventos_participados * 100.0 / e.total_eventos_tipo, 2) AS taxa_presenca_pct
FROM presenca_por_deputado p
JOIN eventos_por_tipo_ano e 
    ON p.descricao_tipo = e.descricao_tipo 
    AND p.ano = e.ano
ORDER BY p.nome_deputado, taxa_presenca_pct DESC

In [0]:
%sql
-- Top 10 deputados com maior taxa de presenca em Sessoes Deliberativas
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    descricao_tipo,
    eventos_participados,
    total_eventos_tipo,
    taxa_presenca_pct
FROM ouro.taxa_presenca
WHERE descricao_tipo = 'Sessão Deliberativa'
ORDER BY taxa_presenca_pct DESC
LIMIT 10

# Entrega 3: Comparativo de Frequência Antes e Depois de Períodos Eleitorais
Compara presença 3 meses antes e 3 meses depois das eleições de outubro de 2022 e 2024.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.comparativo_eleitoral
USING DELTA
COMMENT 'Camada Ouro - Comparativo de frequencia antes e depois das eleicoes de 2024'
AS
SELECT 
    id_deputado,
    nome_deputado,
    sigla_partido,
    sigla_uf,
    CASE 
        WHEN mes BETWEEN 7 AND 9 THEN 'Antes_Eleicao_2024'
        WHEN mes BETWEEN 10 AND 12 THEN 'Depois_Eleicao_2024'
    END AS periodo,
    COUNT(DISTINCT id_evento) AS eventos_participados
FROM prata_presenca_view
WHERE ano = 2024 AND mes BETWEEN 7 AND 12
GROUP BY id_deputado, nome_deputado, sigla_partido, sigla_uf, 
         CASE 
             WHEN mes BETWEEN 7 AND 9 THEN 'Antes_Eleicao_2024'
             WHEN mes BETWEEN 10 AND 12 THEN 'Depois_Eleicao_2024'
         END

In [0]:
%sql
-- Media de eventos por deputado antes e depois das eleicoes
SELECT 
    periodo,
    ROUND(AVG(eventos_participados), 1) AS media_eventos,
    COUNT(DISTINCT id_deputado) AS qtd_deputados
FROM ouro.comparativo_eleitoral
GROUP BY periodo
ORDER BY periodo

# Entrega 4: Densidade de Eventos por Semana e Semanas Sem Atividade
Quantos eventos por semana e quais semanas ficaram sem nenhum evento.


In [0]:
%sql
CREATE OR REPLACE TABLE ouro.densidade_semanal
USING DELTA
COMMENT 'Camada Ouro - Densidade de eventos por semana e semanas sem atividade'
AS
WITH semanas_eventos AS (
    SELECT 
        chave_semana,
        ano,
        semana_ano,
        COUNT(*) AS qtd_eventos,
        COUNT(DISTINCT descricao_tipo) AS tipos_distintos
    FROM prata_eventos_view
    GROUP BY chave_semana, ano, semana_ano
),
todos_anos AS (
    SELECT DISTINCT ano FROM prata_eventos_view
),
semanas_completas AS (
    SELECT 
        a.ano,
        s.semana,
        a.ano * 100 + s.semana AS chave_semana
    FROM todos_anos a
    CROSS JOIN (SELECT explode(sequence(1, 52)) AS semana) s
)
SELECT 
    sc.chave_semana,
    sc.ano,
    sc.semana AS semana_ano,
    COALESCE(se.qtd_eventos, 0) AS qtd_eventos,
    COALESCE(se.tipos_distintos, 0) AS tipos_distintos,
    CASE WHEN se.qtd_eventos IS NULL OR se.qtd_eventos = 0 THEN 'Sem atividade' ELSE 'Com atividade' END AS status
FROM semanas_completas sc
LEFT JOIN semanas_eventos se ON sc.chave_semana = se.chave_semana
ORDER BY sc.chave_semana

In [0]:
%sql
-- Semanas sem atividade
SELECT 
    chave_semana,
    ano,
    semana_ano,
    status
FROM ouro.densidade_semanal
WHERE status = 'Sem atividade'
ORDER BY chave_semana

In [0]:
%sql
SELECT 
    chave_semana,
    ano,
    semana_ano,
    qtd_eventos,
    tipos_distintos
FROM ouro.densidade_semanal
ORDER BY qtd_eventos DESC
LIMIT 10

# Entrega 5: View Pública de Eventos Futuros
Calendário de eventos já agendados (situacao = 'Convocada' ou similar).

In [0]:
%sql
CREATE OR REPLACE VIEW ouro.calendario_eventos_futuros
COMMENT 'Camada Ouro - Exemplo de calendario de eventos (ultimos eventos registrados)'
AS
SELECT 
    id_evento,
    data_hora_inicio,
    data_hora_fim,
    descricao_tipo,
    descricao,
    sigla_orgao,
    nome_orgao,
    local_camara_nome,
    local_externo,
    url_registro
FROM prata.eventos
WHERE data_hora_inicio > '2025-12-01'
ORDER BY data_hora_inicio

In [0]:
%sql
SELECT 
    * 
FROM ouro.calendario_eventos_futuros
limit 10